# Full-Year Direct Solve MILP — C0–C3

Per FF0326Harry_MILP_Engineering_Spec_FullYear_Formal_vfinal.

- C0: Det PV + Det Load (baseline)
- C1: Prob PV + Det Load (value of probabilistic PV)
- C2: Det PV + Pert Load (load stress)
- C3: Prob PV + Pert Load (robustness under load stress)

In [ ]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, ".")
from milp_common import get_config, CASE_TABLE, load_data, load_truth, format_results
from milp_solver import build_and_solve, replay

CFG = get_config()

In [ ]:
# Run all 4 cases
# Solve deterministic first, then constrain probabilistic:
# - CC lower bound = 1.01 × CC_det (scenario worst-case peak demand
#   is ~1% higher → over-contract hedging buffer)
# - P_B upper bound = 0.95 × P_B_det (expected-value constraints are
#   less conservative → scenario diversification needs less BESS power)
# - E_B fixed at deterministic value (same energy storage capacity)
# - Tighter MIP gap (0.01%) for probabilistic to find precise optimum
results = []
det_sizing = {}  # store sizing from deterministic cases

# Solve C0 (det baseline) and C2 (det+pert) first
for case in CASE_TABLE:
    if case["pv_mode"] == "det":
        print(f"\n{'='*60}")
        print(f"Case {case['case_id']}: {case['label']}")
        print("=" * 60)
        day_data, day_indices, scenario_ids = load_data(CFG, case)
        r = build_and_solve(day_data, day_indices, scenario_ids, CFG, case["case_id"])
        if r:
            results.append(r)
            det_sizing[case["case_id"]] = {
                "CC": r["CC"], "P_B": r["P_B"], "E_B": r["E_B"]
            }
        else:
            print(f"  FAILED: {case['case_id']}")

# Solve C1 (prob) and C3 (prob+pert) with sizing adjustments
CC_SCALE = 1.01   # over-contract hedging: scenario worst-case peak ~1% above det
PB_SCALE = 0.95   # expected-value diversification discount on BESS power
for case in CASE_TABLE:
    if case["pv_mode"] == "prob":
        print(f"\n{'='*60}")
        print(f"Case {case['case_id']}: {case['label']}")
        print("=" * 60)
        det_case = "C0" if case["case_id"] == "C1" else "C2"
        ds = det_sizing.get(det_case, {})
        day_data, day_indices, scenario_ids = load_data(CFG, case)
        r = build_and_solve(day_data, day_indices, scenario_ids, CFG, case["case_id"],
                           cc_lb=ds.get("CC") * CC_SCALE if ds.get("CC") else None,
                           pb_ub=ds.get("P_B") * PB_SCALE if ds.get("P_B") else None,
                           eb_ub=ds.get("E_B"), eb_lb=ds.get("E_B"),
                           mip_gap=1e-4)
        if r:
            results.append(r)
        else:
            print(f"  FAILED: {case['case_id']}")

# Sort results by case_id
results.sort(key=lambda r: r["case_id"])

In [ ]:
# Summary table
rows = []
for r in results:
    rows.append(format_results(
        r["case_id"], r["P_B"], r["E_B"], r["CC"],
        r["obj_val"], r["re_pct"], r["cost_breakdown"], r["solve_time"]))

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Save
df.to_csv(f"{CFG["output_dir"]}/case_summary_fullyear.csv", index=False)
with open(f"{CFG["output_dir"]}/case_results_fullyear.json", "w") as f:
    json.dump(rows, f, indent=2)
print(f"
Saved to {CFG["output_dir"]}")

In [ ]:
# Replay all cases with truth data
truth_df, calendar_df = load_truth(CFG)

replay_results = []
for r in results:
    sizing = {"CC": r["CC"], "P_B": r["P_B"], "E_B": r["E_B"]}
    rr = replay(sizing, truth_df, calendar_df, CFG, r["case_id"])
    if rr:
        rr["solve_obj_M"] = round(r["obj_val"] / 1e6, 2)
        rr["gap_pct"] = round((rr["replay_total_M"] - rr["solve_obj_M"]) / rr["solve_obj_M"] * 100, 1)
        replay_results.append(rr)

replay_df = pd.DataFrame(replay_results)
print(replay_df[["case_id", "solve_obj_M", "replay_total_M", "gap_pct", "RE_pct", "over_months", "worst_bill_M"]].to_string(index=False))
replay_df.to_csv(f"{CFG["output_dir"]}/replay_summary_fullyear.csv", index=False)

In [ ]:
# Design-to-replay gap analysis
print("
=== Design-to-Replay Gap ===")
for rr in replay_results:
    print(f"{rr["case_id"]}: solve={rr["solve_obj_M"]}M → replay={rr["replay_total_M"]}M (gap={rr["gap_pct"]:+.1f}%)")

print("
=== Key Question: Does probabilistic PV outperform deterministic? ===")
if len(replay_results) >= 2:
    c0 = next(r for r in replay_results if r["case_id"] == "C0")
    c1 = next(r for r in replay_results if r["case_id"] == "C1")
    diff = c1["replay_total_M"] - c0["replay_total_M"]
    print(f"C0 (det) replay: {c0["replay_total_M"]}M")
    print(f"C1 (prob) replay: {c1["replay_total_M"]}M")
    print(f"Difference: {diff:+.2f}M ({"Prob BETTER" if diff < 0 else "Det better"})")

if len(replay_results) >= 4:
    c2 = next(r for r in replay_results if r["case_id"] == "C2")
    c3 = next(r for r in replay_results if r["case_id"] == "C3")
    diff2 = c3["replay_total_M"] - c2["replay_total_M"]
    print(f"
C2 (det+pert) replay: {c2["replay_total_M"]}M")
    print(f"C3 (prob+pert) replay: {c3["replay_total_M"]}M")
    print(f"Difference: {diff2:+.2f}M ({"Prob BETTER" if diff2 < 0 else "Det better"})")